# 103 - FastAPI: servir modelos con una API

FastAPI es útil cuando una demo Gradio no basta y se necesita exponer un modelo como API consumible por otras aplicaciones.

Este notebook no arranca un servidor real por defecto. Define la API y prueba la lógica de predicción de forma segura dentro del notebook.

## Preparación para Colab

La siguiente celda instala `fastapi` y `uvicorn` solo si se ejecuta en Google Colab y faltan dependencias.

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

def install_if_missing(import_name, package_name):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

if IN_COLAB:
    install_if_missing("fastapi", "fastapi")
    install_if_missing("uvicorn", "uvicorn")
    print("Dependencias de FastAPI preparadas.")
else:
    print("No se ha detectado Colab. Instalación automática omitida.")

## 1. Lógica del modelo separada de la API

Antes de crear endpoints, conviene tener una función de predicción independiente. Aquí usamos reglas sencillas sobre Iris para evitar dependencias pesadas.

In [ ]:
def predict_iris(features):
    sepal_length, sepal_width, petal_length, petal_width = features
    if petal_length < 2.5:
        label = "setosa"
        probabilities = {"setosa": 0.95, "versicolor": 0.04, "virginica": 0.01}
    elif petal_width < 1.8:
        label = "versicolor"
        probabilities = {"setosa": 0.03, "versicolor": 0.84, "virginica": 0.13}
    else:
        label = "virginica"
        probabilities = {"setosa": 0.01, "versicolor": 0.12, "virginica": 0.87}
    return {"prediction": label, "probabilities": probabilities}

print(predict_iris([5.1, 3.5, 1.4, 0.2]))

## 2. Esquemas de entrada y salida

FastAPI usa Pydantic para validar los datos de entrada y documentar automáticamente la API. Si FastAPI no está disponible, se definen clases simples de respaldo.

In [ ]:
try:
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel, Field
    HAS_FASTAPI = True
except Exception:
    FastAPI = None
    HTTPException = Exception
    BaseModel = object
    HAS_FASTAPI = False

if HAS_FASTAPI:
    class IrisInput(BaseModel):
        sepal_length: float = Field(..., ge=0, le=10)
        sepal_width: float = Field(..., ge=0, le=5)
        petal_length: float = Field(..., ge=0, le=7)
        petal_width: float = Field(..., ge=0, le=3)
else:
    class IrisInput:
        def __init__(self, sepal_length, sepal_width, petal_length, petal_width):
            self.sepal_length = sepal_length
            self.sepal_width = sepal_width
            self.petal_length = petal_length
            self.petal_width = petal_width

print("FastAPI disponible:", HAS_FASTAPI)

## 3. Crear la aplicación y endpoints

La API tiene un endpoint de estado y otro de predicción. No se lanza `uvicorn` automáticamente para evitar bloquear el notebook.

In [ ]:
def predict_from_input(data):
    features = [data.sepal_length, data.sepal_width, data.petal_length, data.petal_width]
    return predict_iris(features)

if HAS_FASTAPI:
    app = FastAPI(title="Iris Model API", version="1.0")

    @app.get("/")
    def root():
        return {"status": "ok", "model": "iris_rules_demo"}

    @app.post("/predict")
    def predict_endpoint(data: IrisInput):
        return predict_from_input(data)

    print("App FastAPI creada. En un terminal: uvicorn main:app --reload")
else:
    app = None
    print("FastAPI no disponible. Se usará solo la función predict_from_input.")

## 4. Probar la lógica sin arrancar servidor

En notebooks conviene probar la lógica directamente. Si FastAPI está disponible, también se podría usar `TestClient`.

In [ ]:
sample = IrisInput(sepal_length=6.3, sepal_width=3.3, petal_length=6.0, petal_width=2.5)
print(predict_from_input(sample))

if HAS_FASTAPI:
    try:
        from fastapi.testclient import TestClient
        client = TestClient(app)
        response = client.post("/predict", json={
            "sepal_length": 5.1,
            "sepal_width": 3.5,
            "petal_length": 1.4,
            "petal_width": 0.2,
        })
        print(response.json())
    except Exception as exc:
        print("No se pudo usar TestClient, pero la lógica directa funciona:", exc)

## 5. Comando de despliegue

Para servir esta API fuera del notebook, guarda el código en `main.py` y ejecuta el siguiente comando en terminal.

In [ ]:
print("uvicorn main:app --reload --port 8000")
print("Documentación automática: http://localhost:8000/docs")

## Reto adicional

Añade un endpoint `/model/info` que devuelva nombre del modelo, versión, fecha y variables de entrada.

Para profundizar: consulta `Documentacion/FastAPI_Documentacion.md`.